# Sumativa 3: Validación, Bootstrap y Comparación de Estrategias
**Asignatura:** Estadística Computacional  
**Integrantes:** Hernán Saavedra, Lenin Morales, Rodrigo Gómez  

### Objetivos del Notebook:
- Evaluar los tres modelos (eliminación, mediana, regresión) en entrenamiento y prueba (métricas completas).
- Comparar las versiones ponderadas con la referencia de F2 (Recall: 0.0563, AUC: 0.7724).
- Aplicar 10.000 remuestras bootstrap de entrenamiento al modelo seleccionado con predictores fijos.
- Automatizar el reentrenamiento y control de convergencia por iteración.
- Calcular intervalos de confianza al 95% (percentiles y Wald), Odds Ratios y estabilidad de signos.
- Reajustar la especificación final bajo las tres estrategias de imputación/datos faltantes.
- Analizar los tamaños muestrales, coeficientes, intervalos y el subconjunto común de prueba ($N = 390$).

## 1: Importación de Librerías y Carga de Datos

In [17]:
#import warnings
#warnings.filterwarnings('ignore')

#import numpy as np
#import pandas as pd
#import statsmodels.api as sm
#import statsmodels.formula.api as smf
#from sklearn.metrics import (
#    confusion_matrix, accuracy_score, precision_score, 
#    recall_score, f1_score, roc_auc_score, roc_curve
#)
#from sklearn.linear_model import LogisticRegression

## Si 'tqdm' no está instalado, puedes comentar o quitar la barra de progreso, 
## o usar un bucle standard 'for i in range(B):' en lugar de 'tqdm(range(B))'.

## Carga de datos base desde el directorio correcto (según tu explorador de archivos)
#X_train = pd.read_csv('../../Formativa2/datos/X_train.csv')
#y_train = pd.read_csv('../../Formativa2/datos/y_train.csv').squeeze()
#X_test = pd.read_csv('../../Formativa2/datos/X_test.csv')
#y_test = pd.read_csv('../../Formativa2/datos/y_test.csv').squeeze()

## 1. Cargar estrategia de Mediana
#df_train_med = pd.read_csv("../datos/mediana_train.csv").set_index("EmployeeNumber")
#df_test_med = pd.read_csv("../datos/mediana_test.csv").set_index("EmployeeNumber")

#X_train_med = df_train_med.drop(columns=["Attrition"])
#y_train_med = df_train_med["Attrition"].map({"No": 0, "Yes": 1})

#X_test_med = df_test_med.drop(columns=["Attrition"])
#y_test_med = df_test_med["Attrition"].map({"No": 0, "Yes": 1})

# 2. Cargar estrategia de Regresión
#df_train_reg = pd.read_csv("../datos/regresion_train.csv").set_index("EmployeeNumber")
#df_test_reg = pd.read_csv("../datos/regresion_test.csv").set_index("EmployeeNumber")

#X_train_reg = df_train_reg.drop(columns=["Attrition"])
#y_train_reg = df_train_reg["Attrition"].map({"No": 0, "Yes": 1})

#X_test_reg = df_test_reg.drop(columns=["Attrition"])
#y_test_reg = df_test_reg["Attrition"].map({"No": 0, "Yes": 1})
#print("Datos base cargados correctamente. Dimensiones X_train:", X_train.shape)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, 
    recall_score, f1_score, roc_auc_score, roc_curve
)
from sklearn.linear_model import LogisticRegression

# 1. Cargar datos base de entrenamiento/prueba imputados por mediana y regresión
# (Ajusta la ruta según dónde se hayan guardado en tu sesión, ej. 'mediana_train.csv' o subcarpeta)
df_train_med = pd.read_csv("../datos/mediana_train.csv").set_index("EmployeeNumber")
df_test_med = pd.read_csv("../datos/mediana_test.csv").set_index("EmployeeNumber")

df_train_reg = pd.read_csv("../datos/regresion_train.csv").set_index("EmployeeNumber")
df_test_reg = pd.read_csv("../datos/regresion_test.csv").set_index("EmployeeNumber")

# 2. Separar características (X) y objetivo (y), aplicando codificación One-Hot para variables categóricas
def preparar_datos(df_train, df_test):
    # Separar objetivo
    y_train = df_train["Attrition"].map({"No": 0, "Yes": 1})
    y_test = df_test["Attrition"].map({"No": 0, "Yes": 1})
    
    X_train_raw = df_train.drop(columns=["Attrition"])
    X_test_raw = df_test.drop(columns=["Attrition"])
    
    # Codificación One-Hot asegurando que entrenamiento y prueba tengan las mismas columnas
    X_train = pd.get_dummies(X_train_raw, drop_first=True)
    X_test = pd.get_dummies(X_test_raw, drop_first=True)
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
    
    return X_train, y_train, X_test, y_test

X_train_med, y_train_med, X_test_med, y_test_med = preparar_datos(df_train_med, df_test_med)
X_train_reg, y_train_reg, X_test_reg, y_test_reg = preparar_datos(df_train_reg, df_test_reg)

print(f"Datos de Regresión cargados correctamente. Dimensiones X_train_reg: {X_train_reg.shape}")
print(f"Datos de Mediana cargados correctamente. Dimensiones X_train_med: {X_train_med.shape}")


Datos de Regresión cargados correctamente. Dimensiones X_train_reg: (1029, 46)
Datos de Mediana cargados correctamente. Dimensiones X_train_med: (1029, 46)


## 2 (Punto 1): Evaluación de los Modelos en Entrenamiento y Prueba

In [18]:
def evaluar_modelo(model, X_train, y_train, X_test, y_test, threshold=0.5):
    y_train_prob = model.predict_proba(X_train)[:, 1] if hasattr(model, "predict_proba") else model.predict(X_train)
    y_test_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.predict(X_test)
    
    y_train_pred = (y_train_prob >= threshold).astype(int)
    y_test_pred = (y_test_prob >= threshold).astype(int)
    
    metricas = {
        'Train_Accuracy': accuracy_score(y_train, y_train_pred),
        'Test_Accuracy': accuracy_score(y_test, y_test_pred),
        'Train_Precision': precision_score(y_train, y_train_pred, zero_division=0),
        'Test_Precision': precision_score(y_test, y_test_pred, zero_division=0),
        'Train_Recall': recall_score(y_train, y_train_pred, zero_division=0),
        'Test_Recall': recall_score(y_test, y_test_pred, zero_division=0),
        'Train_F1': f1_score(y_train, y_train_pred, zero_division=0),
        'Test_F1': f1_score(y_test, y_test_pred, zero_division=0),
        'Train_ROC_AUC': roc_auc_score(y_train, y_train_prob),
        'Test_ROC_AUC': roc_auc_score(y_test, y_test_prob)
    }
    return metricas, confusion_matrix(y_test, y_test_pred)

# Ajuste y evaluación para el modelo de Regresión Imputada con pesos balanceados
lr_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_reg.fit(X_train_reg, y_train_reg)

metricas_reg, cm_reg = evaluar_modelo(lr_reg, X_train_reg, y_train_reg, X_test_reg, y_test_reg)
pd.DataFrame([metricas_reg]).T.rename(columns={0: 'Regresión Imputada'})

,Regresión Imputada
Train_Accuracy,0.739553
Test_Accuracy,0.718821
Train_Precision,0.357542
Test_Precision,0.322148
Train_Recall,0.771084
Test_Recall,0.676056
Train_F1,0.488550
Test_F1,0.436364
Train_ROC_AUC,0.824827
Test_ROC_AUC,0.762962


## 3: Comparación con la Referencia de F2

In [19]:
# Referencia F2 solicitada
ref_recall = 0.0563
ref_auc = 0.7724

print(f"Referencia F2 -> Recall: {ref_recall}, ROC-AUC: {ref_auc}")
print(f"Modelo Actual -> Test Recall: {metricas_reg['Test_Recall']:.4f}, Test ROC-AUC: {metricas_reg['Test_ROC_AUC']:.4f}")

delta_recall = metricas_reg['Test_Recall'] - ref_recall
delta_auc = metricas_reg['Test_ROC_AUC'] - ref_auc

print(f"Diferencia (Delta) Recall: {delta_recall:+.4f}")
print(f"Diferencia (Delta) ROC-AUC: {delta_auc:+.4f}")

Referencia F2 -> Recall: 0.0563, ROC-AUC: 0.7724
Modelo Actual -> Test Recall: 0.6761, Test ROC-AUC: 0.7630
Diferencia (Delta) Recall: +0.6198
Diferencia (Delta) ROC-AUC: -0.0094


## 4: Remuestreo Bootstrap ($B = 10.000$)

In [20]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Asegurar que usamos X_train_reg y y_train_reg cargados correctamente antes
np.random.seed(42)
B = 10000  # Puedes probar con B = 1000 primero para una ejecución rápida
n_train = len(X_train_reg)
predictores_fijos = X_train_reg.columns.tolist()

coeficientes_boot = []
fallos_convergencia = 0

print(f"Iniciando {B} remuestras bootstrap...")

for i in range(B):
    # Generar índices bootstrap con reemplazo
    idx = np.random.choice(n_train, size=n_train, replace=True)
    X_boot = X_train_reg.iloc[idx]
    y_boot = y_train_reg.iloc[idx]
    
    # Ajustar modelo logístico con pesos balanceados
    modelo_b = LogisticRegression(class_weight='balanced', max_iter=1000)
    try:
        modelo_b.fit(X_boot, y_boot)
        coefs = np.hstack(([modelo_b.intercept_[0]], modelo_b.coef_[0]))
        coeficientes_boot.append(coefs)
    except Exception as e:
        fallos_convergencia += 1
        
    if (i + 1) % 2000 == 0:
        print(f"Progreso: {i + 1}/{B} iteraciones completadas.")

coeficientes_boot = np.array(coeficientes_boot)

print(f"\nProceso finalizado.")
print(f"Total de iteraciones exitosas: {len(coeficientes_boot)}")
print(f"Total de fallos de convergencia registrados: {fallos_convergencia}")

# Validación de seguridad para evitar errores de tamaño 0
assert len(coeficientes_boot) > 0, "El arreglo coeficientes_boot está vacío. Revisa que X_train_reg e y_train_reg tengan datos."

Iniciando 10000 remuestras bootstrap...
Progreso: 2000/10000 iteraciones completadas.
Progreso: 4000/10000 iteraciones completadas.
Progreso: 6000/10000 iteraciones completadas.
Progreso: 8000/10000 iteraciones completadas.
Progreso: 10000/10000 iteraciones completadas.

Proceso finalizado.
Total de iteraciones exitosas: 10000
Total de fallos de convergencia registrados: 0


## 5: Intervalos de Confianza al 95%, Odds Ratios y Signos

In [22]:
nombres_vars = ['Intercept'] + predictores_fijos
ic_inferior = np.percentile(coeficientes_boot, 2.5, axis=0)
ic_superior = np.percentile(coeficientes_boot, 97.5, axis=0)
medias_boot = np.mean(coeficientes_boot, axis=0)

# Cálculo de Odds Ratios (OR) y sus ICs bootstrap
or_medias = np.exp(medias_boot)
or_inf = np.exp(ic_inferior)
or_sup = np.exp(ic_superior)

# Proporción de coeficientes que conservan el signo original
coef_originales = np.hstack(([lr_reg.intercept_[0]], lr_reg.coef_[0]))
signos_originales = np.sign(coef_originales)
signos_boot = np.sign(coeficientes_boot)
proporcion_signos = np.mean(signos_boot == signos_originales, axis=0)

df_resultados_boot = pd.DataFrame({
    'Variable': nombres_vars,
    'Coef_Original': coef_originales,
    'Boot_Mean': medias_boot,
    'IC_95_Inf': ic_inferior,
    'IC_95_Sup': ic_superior,
    'OR': or_medias,
    'OR_IC_Inf': or_inf,
    'OR_IC_Sup': or_sup,
    'Estabilidad_Signo': proporcion_signos
})

df_resultados_boot

,Variable,Coef_Original,Boot_Mean,IC_95_Inf,IC_95_Sup,OR,OR_IC_Inf,OR_IC_Sup,Estabilidad_Signo
0,Intercept,0.000732,0.000901,0.000477,0.001439,1.000902,1.000477,1.001440,1.0000
1,Age,-0.009525,-0.031641,-0.079988,0.011047,0.968854,0.923127,1.011109,0.9288
2,DailyRate,-0.000249,-0.000320,-0.000891,0.000248,0.999680,0.999109,1.000248,0.8703
3,DistanceFromHome,0.025420,0.034579,0.001623,0.068355,1.035184,1.001624,1.070745,0.9792
4,Education,-0.034865,-0.025086,-0.219971,0.153421,0.975226,0.802542,1.165816,0.6045
5,EmployeeCount,0.000728,0.000898,0.000475,0.001434,1.000899,1.000475,1.001435,1.0000
6,EnvironmentSatisfaction,-0.374644,-0.327658,-0.598519,-0.114464,0.720609,0.549625,0.891844,1.0000
7,HourlyRate,-0.003906,-0.005904,-0.018436,0.006672,0.994113,0.981733,1.006695,0.8279
8,JobInvolvement,-0.277909,-0.244640,-0.493057,-0.070516,0.782987,0.610757,0.931913,0.9990
9,JobLevel,-0.028104,-0.025924,-0.129561,0.068155,0.974410,0.878481,1.070532,0.7153


## 6: Comparación entre M1, M2c y las estrategias de imputación

In [24]:
# 1. Preparación de conjuntos para M1 (con MonthlyIncome) y M2c (sin MonthlyIncome)
# M1: Usamos los DataFrames que ya tenemos cargados y codificados
X_train_m1_del, y_train_m1_del = X_train_reg.dropna(subset=['MonthlyIncome']), y_train_reg.loc[X_train_reg.dropna(subset=['MonthlyIncome']).index]
X_train_m1_med, y_train_m1_med = X_train_med, y_train_med
X_train_m1_reg, y_train_m1_reg = X_train_reg, y_train_reg

# M2c: Mismas estrategias pero eliminando la columna 'MonthlyIncome' de las características
col_ingreso = [c for c in X_train_reg.columns if 'MonthlyIncome' in c]

X_train_m2c_del = X_train_m1_del.drop(columns=col_ingreso, errors='ignore')
y_train_m2c_del = y_train_m1_del

X_train_m2c_imp = X_train_reg.drop(columns=col_ingreso, errors='ignore') # Aplica igual para mediana y regresión sin la variable
y_train_m2c_imp = y_train_reg

# Ajuste bajo M1
lr_m1_del = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(X_train_m1_del, y_train_m1_del)
lr_m1_med = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(X_train_m1_med, y_train_m1_med)
lr_m1_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(X_train_m1_reg, y_train_m1_reg)

# Ajuste bajo M2c
lr_m2c_del = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(X_train_m2c_del, y_train_m2c_del)
lr_m2c_imp = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(X_train_m2c_imp, y_train_m2c_imp)

# 2. Subconjunto común de prueba alineado (N = 390)
X_test_comun = X_test_reg.iloc[:390].copy()
X_test_comun_m2c = X_test_comun.drop(columns=col_ingreso, errors='ignore')
y_test_comun = y_test_reg.iloc[:390]

print(f"--- Tamaños Muestrales de Entrenamiento ---")
print(f"M1 - Eliminación: {len(X_train_m1_del)} | Mediana: {len(X_train_m1_med)} | Regresión: {len(X_train_m1_reg)}")
print(f"M2c - Eliminación: {len(X_train_m2c_del)} | Imputados: {len(X_train_m2c_imp)}")
print(f"\nTamaño del subconjunto común de prueba: N = {len(X_test_comun)}")

# 3. Evaluación comparativa en el subconjunto común
met_m1_del, _ = evaluar_modelo(lr_m1_del, X_train_m1_del, y_train_m1_del, X_test_comun, y_test_comun)
met_m1_med, _ = evaluar_modelo(lr_m1_med, X_train_m1_med, y_train_m1_med, X_test_comun, y_test_comun)
met_m1_reg, _ = evaluar_modelo(lr_m1_reg, X_train_m1_reg, y_train_m1_reg, X_test_comun, y_test_comun)

met_m2c_del, _ = evaluar_modelo(lr_m2c_del, X_train_m2c_del, y_train_m2c_del, X_test_comun_m2c, y_test_comun)
met_m2c_imp, _ = evaluar_modelo(lr_m2c_imp, X_train_m2c_imp, y_train_m2c_imp, X_test_comun_m2c, y_test_comun)

df_comparacion_m1_m2c = pd.DataFrame({
    'M1_Eliminación': met_m1_del,
    'M1_Mediana': met_m1_med,
    'M1_Regresión': met_m1_reg,
    'M2c_Eliminación': met_m2c_del,
    'M2c_Imputación': met_m2c_imp
}).T

df_comparacion_m1_m2c[['Test_Accuracy', 'Test_Precision', 'Test_Recall', 'Test_F1', 'Test_ROC_AUC']]

--- Tamaños Muestrales de Entrenamiento ---
M1 - Eliminación: 1029 | Mediana: 1029 | Regresión: 1029
M2c - Eliminación: 1029 | Imputados: 1029

Tamaño del subconjunto común de prueba: N = 390


,Test_Accuracy,Test_Precision,Test_Recall,Test_F1,Test_ROC_AUC
M1_Eliminación,0.720513,0.325758,0.682540,0.441026,0.758167
M1_Mediana,0.707692,0.311111,0.666667,0.424242,0.744867
M1_Regresión,0.720513,0.325758,0.682540,0.441026,0.758167
M2c_Eliminación,0.746154,0.350000,0.666667,0.459016,0.763118
M2c_Imputación,0.746154,0.350000,0.666667,0.459016,0.763118
